**GPU note:** T4 is the confirmed-working fallback for this notebook. On **A100**, training crashes with a CUDA "illegal memory access" in Longformer's legacy global-attention code (`_compute_attn_output_with_global_indices`, the `attn_probs.narrow(...)` call) even at `per_device_train_batch_size=1`. Ruled out so far: batch size, bf16/fp16, gradient_checkpointing, TF32, and attention implementation (eager vs auto) — none of these fixed it.

Current experiment: **pinning an older `torch` version** (see the pip install cell below), on the theory that Longformer's `as_strided()`/`unfold()`-based sliding-window attention is sensitive to CUDA-kernel-selection behavior that changed in a later PyTorch release, and that this only surfaces on Ampere (A100) and not Turing (T4). **This is untested.**

Important: after changing the pinned `torch` version, you must **fully restart the Colab runtime** (Runtime → Restart session) before re-running — `pip install`ing a different torch build mid-session will not take effect while the old one is already imported, and if a prior cell already crashed with an illegal memory access, the CUDA context is poisoned and every subsequent CUDA call will fail the same way regardless of what code runs, so a stale runtime will always look like the bug persists even if the fix worked.

In [ ]:
!pip uninstall -y torchvision torchaudio torchao -q
!pip install "transformers==4.46.3" "torch==2.2.2" "numpy<2" datasets==2.19.0 accelerate evaluate scipy scikit-learn --quiet
!pip install --force-reinstall --no-deps -q pandas pyarrow

import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("masharma/convolearn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "allenai/longformer-base-4096"
)

def count_tokens(example):
    return {
        "num_tokens": len(
            tokenizer(
                example["cleaned_conversation"],
                truncation=False
            )["input_ids"]
        )
    }

token_lengths = dataset["train"].map(count_tokens)

In [5]:
import numpy as np

lengths = np.array(token_lengths["num_tokens"])
print(f"mean: {lengths.mean():.0f}, median: {np.median(lengths):.0f}, max: {lengths.max()}")
print(f"share of conversations > 4096 tokens: {(lengths > 4096).mean():.2%}")

mean: 708, median: 641, max: 13549
share of conversations > 4096 tokens: 0.14%


In [6]:
from datasets import DatasetDict
import numpy as np

raw = dataset["train"]

# Group row indices by topic
topic_to_indices = {}
for i, topic in enumerate(raw["earthscience_topic"]):
    if topic not in topic_to_indices:
        topic_to_indices[topic] = []
    topic_to_indices[topic].append(i)

print("Topic distribution:")
for topic, indices in topic_to_indices.items():
    print(f"  {topic}: {len(indices)} rows")

# For each topic, split its rows 70/15/15
train_indices, val_indices, test_indices = [], [], []

rng = np.random.default_rng(seed=42)

for topic, indices in topic_to_indices.items():
    indices = np.array(indices)
    rng.shuffle(indices)

    n = len(indices)
    n_train = int(0.70 * n)
    n_val   = int(0.15 * n)

    train_indices.extend(indices[:n_train].tolist())
    val_indices.extend(indices[n_train : n_train + n_val].tolist())
    test_indices.extend(indices[n_train + n_val :].tolist())

# Build splits
train_ds = raw.select(train_indices)
val_ds   = raw.select(val_indices)
test_ds  = raw.select(test_indices)

dataset_dict = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})


Topic distribution:
  Earth's Energy: 794 rows
  Investigation and Experimentation: 365 rows
  Solid Earth: 371 rows
  Astronomy and Cosmology: 604 rows


In [7]:
print("Unique values in 'kb_dim':", raw.unique('kb_dim'))
print("Unique values in 'kb_subdim':", raw.unique('kb_subdim'))

Unique values in 'kb_dim': ['Metacognition', 'Formative Assessment', 'Cognitive Engagement', 'Accountability', 'Cultural Responsiveness', 'Power Dynamics']
Unique values in 'kb_subdim': ['Reflective Growth', 'Strategic Thinking', 'Continuous Assessment', 'Scaffolding', 'Evidence-Based Reasoning', 'Self-Assessment', 'Moral Responsibility', 'Depth of Reasoning', 'Error Analysis', 'Problem-Based Learning', 'Cultural Analogies', 'Critical Thinking', 'Thinking Aloud', 'Ownership of Ideas', 'Generative Questioning', 'Partisanship', 'Reflection', 'Self-Reflection', 'Synthesizing', 'Cultural Identity Exploration', 'Persuasive Discourse']


In [8]:
def format_conversation(example):
    text = (
        f"Dimension: {example['kb_dim']}\n"
        f"{example['cleaned_conversation']}"
    )
    return {"input_text": text}

dataset_dict = dataset_dict.map(format_conversation)

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")

def tokenize(example):
    encoding = tokenizer(
        example["input_text"],
        max_length=4096,
        truncation=True,
        padding=False,  # no padding here — done per batch instead
    )
    encoding["global_attention_mask"] = [0] * len(encoding["input_ids"])
    encoding["global_attention_mask"][0] = 1
    encoding["labels"] = float(example["effectiveness_consensus"])
    return encoding

tokenized = dataset_dict.map(
    tokenize,
    remove_columns=dataset_dict["train"].column_names,
)

print(tokenized)
print("Sample keys:", list(tokenized["train"].features.keys()))
print("Labels (first 3):", tokenized["train"]["labels"][:3])

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels'],
        num_rows: 1491
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels'],
        num_rows: 318
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels'],
        num_rows: 325
    })
})
Sample keys: ['input_ids', 'attention_mask', 'global_attention_mask', 'labels']
Labels (first 3): [2.5, 4.5, 4.0]


In [10]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from scipy.stats import pearsonr, spearmanr
import numpy as np
import torch

# A100: disable TF32 (Ampere-only, could be interacting badly with Longformer's
# dynamic-shape global-attention indexing) and force eager attention to rule out
# a fused/SDPA attention path being silently selected.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/longformer-base-4096",
    num_labels=1,
    problem_type="regression",
    attn_implementation="eager",
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.squeeze(preds)
    mse = np.mean((preds - labels) ** 2)
    pearson = pearsonr(preds, labels)[0]
    spearman = spearmanr(preds, labels)[0]
    return {"mse": mse, "pearson": pearson, "spearman": spearman}

# A100 config: batch_size stays at 1, bf16 (no gradient_checkpointing needed —
# A100 has ample memory for a single sequence, and native bf16 tensor cores).
training_args = TrainingArguments(
    output_dir="./longformer-convolearn",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    seed=42,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Epoch,Training Loss,Validation Loss


RuntimeError: Numpy is not available

In [ ]:
test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
print(test_metrics)

In [ ]:
model_path = './final_model_checkpoint'

# Save the model and tokenizer to the local directory
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model and tokenizer saved to {model_path}")

## Stage 2: multi-head model for individual per-dimension scores

Shared Longformer encoder + one linear output per `kb_dim` (6 heads total), trained with masked MSE so each example only updates the head matching its own dimension. No dimension-conditioning text needed — raw `cleaned_conversation` in, all 6 dimension scores out in a single forward pass. Reporting focuses on Cognitive Engagement, Formative Assessment, and Metacognition, but all 6 heads are trained since it costs nothing and helps the shared encoder generalize.

In [ ]:
ALL_DIMS = sorted(raw.unique("kb_dim"))
DIM2IDX = {dim: i for i, dim in enumerate(ALL_DIMS)}
TARGET_DIMS = ["Cognitive Engagement", "Formative Assessment", "Metacognition"]

print("Dimension -> head index:", DIM2IDX)

def tokenize_multihead(example):
    encoding = tokenizer(
        example["cleaned_conversation"],
        max_length=4096,
        truncation=True,
        padding=False,
    )
    encoding["global_attention_mask"] = [0] * len(encoding["input_ids"])
    encoding["global_attention_mask"][0] = 1
    encoding["labels"] = float(example["effectiveness_consensus"])
    encoding["dim_idx"] = DIM2IDX[example["kb_dim"]]
    return encoding

tokenized_mh = dataset_dict.map(
    tokenize_multihead,
    remove_columns=dataset_dict["train"].column_names,
)

print(tokenized_mh)
print("Sample dim_idx (first 5):", tokenized_mh["train"]["dim_idx"][:5])

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import LongformerModel

class MultiHeadLongformerRegressor(nn.Module):
    def __init__(self, base_model_name, num_dims):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(base_model_name, attn_implementation="eager")
        self.heads = nn.Linear(self.longformer.config.hidden_size, num_dims)

    def gradient_checkpointing_enable(self, **kwargs):
        self.longformer.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.longformer.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, global_attention_mask, dim_idx=None, labels=None):
        outputs = self.longformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask,
        )
        pooled = outputs.last_hidden_state[:, 0]  # CLS token representation
        logits = self.heads(pooled)  # (batch, num_dims) — one score per dimension

        loss = None
        if labels is not None and dim_idx is not None:
            selected = logits.gather(1, dim_idx.unsqueeze(1)).squeeze(1)
            loss = F.mse_loss(selected, labels)

        return {"loss": loss, "logits": logits}

mh_model = MultiHeadLongformerRegressor("allenai/longformer-base-4096", num_dims=len(ALL_DIMS))

In [ ]:
def compute_metrics_multihead(eval_pred):
    preds = eval_pred.predictions  # (N, num_dims)
    labels = np.array(eval_pred.label_ids)  # (N,)
    dim_idx = np.array(eval_pred.inputs["dim_idx"])  # (N,)
    selected = preds[np.arange(len(labels)), dim_idx]

    metrics = {
        "overall_mse": float(np.mean((selected - labels) ** 2)),
        "overall_pearson": float(pearsonr(selected, labels)[0]),
    }
    for i, name in enumerate(ALL_DIMS):
        mask = dim_idx == i
        if mask.sum() > 1:
            slug = name.lower().replace(" ", "_")
            metrics[f"{slug}_mse"] = float(np.mean((selected[mask] - labels[mask]) ** 2))
            metrics[f"{slug}_pearson"] = float(pearsonr(selected[mask], labels[mask])[0])
    return metrics

# A100 config: batch_size stays at 1 (see GPU note above), bf16, no gradient_checkpointing.
mh_training_args = TrainingArguments(
    output_dir="./longformer-convolearn-multihead",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_overall_mse",
    greater_is_better=False,
    logging_steps=50,
    seed=42,
    include_inputs_for_metrics=True,
    label_names=["labels"],
)

mh_trainer = Trainer(
    model=mh_model,
    args=mh_training_args,
    train_dataset=tokenized_mh["train"],
    eval_dataset=tokenized_mh["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_multihead,
)

mh_trainer.train()

In [ ]:
mh_test_metrics = mh_trainer.evaluate(tokenized_mh["test"], metric_key_prefix="test")
print(mh_test_metrics)